# Quick start
The `batcamp` package builds an [octree](https://en.wikipedia.org/wiki/Octree) over known mesh cell geometry, and uses using that octree for fast resampling and related spatial queries on scientific simulation data. The library is written to support fast analysis of [SWMF/BATSRUS](https://clasp.engin.umich.edu/research/theory-computational-methods/space-weather-modeling-framework/) data for which only point and hexahedron information is stored. `batcamp` uses the hexahedral "leaf cells" to reconstruct a full, searchable octree structure.

`batcamp` provides an octree based interpolator which outperforms the `scipy` workhorse [`LinearNDInterpolator`](https://scipy.github.io/devdocs/reference/generated/scipy.interpolate.LinearNDInterpolator.html), as it makes full use of the octree structure of the data.

## Local example
A sample BATSRUS file ships with the repository. It contains point and leaf cell data. We read it with the `batread` module.

In [ ]:
from batread import Dataset

ds = Dataset.from_file("../sample_data/3d__var_1_n00000000.plt")
print(ds)

The octree can now be built using the point and corner data. The $xyz$ coordinates comprise the first three variables, and the rest are field values. The corner_ids index into the point data; each cell comprises eight corners.

In [ ]:
points_xyz = ds.points[..., :3]
points_data = ds.points[..., 3:]
print(f"Point coordinates: {points_xyz.shape=}")
print(f"Point data: {points_data.shape=}")
corner_ids = ds.corners
print(f"Corner IDs: {corner_ids.shape=}")

Here we show how the octree is built from points and corner IDs. There is also a convenience method that builds the octree directly from the `ds` (shown later).

In [ ]:
from batcamp import Octree
octree = Octree(points_xyz, corner_ids)
print(octree)

With the octree reconstructed, it can be fed to the octree interpolator.

In [ ]:
from batcamp import OctreeInterpolator
interp = OctreeInterpolator(octree, points_data)
print(interp)

A simple first query is a Cartesian slice through `z = 0`. The interpolator accepts grid-shaped `X`, `Y`, `Z` arrays directly, so the plotting code can stay close to the mathematics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
X, Y = np.meshgrid(np.linspace(-24, 24, 512), np.linspace(-24, 24, 512))
Z = np.zeros_like(X)
data = interp(X, Y, Z)

fig, ax = plt.subplots()
pcm = ax.pcolormesh(X, Y, data[..., 0], norm="log")
fig.colorbar(pcm, ax=ax)
ax.set_title("Midplane slice of Rho [g/cm^3]")
plt.show()

## Two larger examples
The next examples use one solar corona file and one inner heliosphere file from a larger archive. These data files contain many variables other than the density-only example above. The data files are downloaded and cached with the `pooch` package.

In [ ]:
import pooch

[sc_file] = pooch.retrieve(
    url = "https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz",
    known_hash="c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136",
    progressbar=True,
    processor=pooch.Untar(
        members = ["run-Sun-G2211/SC/IO2/3d__var_4_n00044000.plt"]
    ),
)

print(f"{sc_file=}")

## Solar corona (SC)
The solar-corona dataset uses a spherically structured octree; this is, however, transparent to the user.

In [ ]:
sc_ds = Dataset.from_file(sc_file)
print(sc_ds)

In [ ]:
sc_octree = Octree.from_ds(sc_ds)  # This is a convenience method that does the same thing as Octree(point_xyz, corner_ids)
print(sc_octree)

In [ ]:
sc_interp = OctreeInterpolator(sc_octree, sc_ds["Rho [g/cm^3]"])  # Named indexing is used on the ds object.
print(sc_interp)

In [ ]:
sc_X, sc_Y = np.meshgrid(np.linspace(-60, 60, 512), np.linspace(-60, 60, 512))
sc_Z = np.zeros_like(sc_X)
sc_rho = sc_interp(sc_X, sc_Y, sc_Z)

fig, ax = plt.subplots()
pcm = ax.pcolormesh(sc_X, sc_Y, sc_rho, norm="log")
fig.colorbar(pcm, ax=ax).set_label("Rho [g/cm^3]")
ax.set_title("Density in midplane $z=0$")

plt.show()

## Inner heliosphere (IH)
The inner-heliosphere file is a cartesian octree.

In [ ]:
[ih_file] = pooch.retrieve(
    url = "https://zenodo.org/records/7110555/files/run-Sun-G2211.tar.gz",
    known_hash="c31a32aab08cc20d5b643bba734fd7220e6b369e691f55f88a3a08cc5b2a2136",
    progressbar=True,
    processor=pooch.Untar(
        members = ["run-Sun-G2211/IH/IO2/3d__var_4_n00005000.plt"]
    ),
)

print(f"{ih_file=}")

In [ ]:
ih_ds = Dataset.from_file(ih_file)
print(ih_ds)

In [ ]:
ih_octree = Octree.from_ds(ih_ds)
print(ih_octree)

In [ ]:
ih_interp = OctreeInterpolator(ih_octree, ih_ds["Rho [g/cm^3]"])
print(ih_interp)

The hole in the middle of the IH domain should be filled with `sc_interp` data in a full analysis, but here we just show the IH data on its own.

In [ ]:
ih_X, ih_Y = np.meshgrid(np.linspace(-200, 200, 512), np.linspace(-200, 200, 512))
ih_Z = np.zeros_like(ih_X)
ih_rho = ih_interp(ih_X, ih_Y, ih_Z)

fig, ax = plt.subplots()
pcm = ax.pcolormesh(ih_X, ih_Y, ih_rho, norm="log")
fig.colorbar(pcm, ax=ax).set_label("Rho [g/cm^3]")
ax.set_title("Density in midplane $z=0$")
plt.show()

## Resample onto a sphere
Here the solar corona data is sampled onto a sphere.

In [ ]:
_n = 360*2
pp, aa = np.meshgrid(
    np.linspace(0.0, np.pi, _n),
    np.linspace(0.0, 2.0 * np.pi, 2 * _n), indexing="xy")

r_sphere = 20.0
rr = r_sphere * np.ones_like(aa)

In [ ]:
sphere_xyz = np.stack((
    rr * np.sin(pp) * np.cos(aa),
    rr * np.sin(pp) * np.sin(aa),
    rr * np.cos(pp)), axis=-1)

sphere_data, cell_sphere = sc_interp(sphere_xyz, return_cell_ids=True)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
im = ax.pcolormesh(np.degrees(aa), np.degrees(pp), sphere_data, norm="log")
ax.set_xlabel("azimuth [deg]")
ax.set_ylabel("polar [deg]")
ax.set_title(f"Density on sphere r={r_sphere} R_Sun")
cb = fig.colorbar(im, ax=ax)
cb.set_label("Rho [g/cm^3]")
ax.invert_yaxis()
plt.show()

For the inner heliosphere, we can do the same thing. We just need to make sure to use the correct octree interpolator (`ih_interp` instead of `sc_interp`) and to use a larger sphere radius.

In [ ]:
r_sphere = 215.0
sphere_xyz = np.stack((
    r_sphere * np.sin(pp) * np.cos(aa),
    r_sphere * np.sin(pp) * np.sin(aa),
    r_sphere * np.cos(pp)), axis=-1)

sphere_data, cell_sphere = ih_interp(sphere_xyz, return_cell_ids=True)
sphere_data.shape

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
im = ax.pcolormesh(np.degrees(aa), np.degrees(pp), sphere_data, norm="log")
ax.set_xlabel("azimuth [deg]")
ax.set_ylabel("polar [deg]")
ax.set_title(f"Density on sphere r={r_sphere} R_Sun")
cb = fig.colorbar(im, ax=ax)
cb.set_label("Rho [g/cm^3]")
ax.invert_yaxis()
plt.show()